## AI-Powered Sustainable Multimodal Route Planner

### 1. Project Overview

This project develops an AI-powered sustainable multimodal route planner
that recommends feasible journeys based on a user's maximum budget,
maximum travel time, and available transportation modes.

The system generates routes using connected transportation segments and
allows different modes to be used across different segments of the same
journey. It evaluates total travel time, cost, and CO₂ emissions to
recommend suitable route options.

The project focuses on sustainable urban mobility and supports SDG 11:
Sustainable Cities and Communities, with SDG 13: Climate Action as a
secondary alignment.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('multimodal_mobility_network.csv')
df

,connection_id,source,destination,mode,distance_km,duration_min,cost_inr,co2_kg
0,C001,Koti,Abids,Walking,2.0,26,0,0.0
1,C001,Koti,Abids,Bicycle,2.0,9,0,0.0
2,C001,Koti,Abids,Public Transport,2.0,16,15,0.2
3,C001,Koti,Abids,Car,2.0,8,35,0.5
4,C002,Koti,Nampally,Walking,3.0,38,0,0.0
...,...,...,...,...,...,...,...,...
115,C029,Tolichowki,Gachibowli,Car,5.0,17,85,1.0
116,C030,HITEC City,Kukatpally,Walking,7.0,89,0,0.0
117,C030,HITEC City,Kukatpally,Bicycle,7.0,30,0,0.0
118,C030,HITEC City,Kukatpally,Public Transport,7.0,50,25,0.7


In [3]:
df.head()

,connection_id,source,destination,mode,distance_km,duration_min,cost_inr,co2_kg
0,C001,Koti,Abids,Walking,2.0,26,0,0.0
1,C001,Koti,Abids,Bicycle,2.0,9,0,0.0
2,C001,Koti,Abids,Public Transport,2.0,16,15,0.2
3,C001,Koti,Abids,Car,2.0,8,35,0.5
4,C002,Koti,Nampally,Walking,3.0,38,0,0.0


#### data understanding and cleaning

In [4]:
print("Dataset Shape:", df.shape)

Dataset Shape: (120, 8)


In [5]:
print("Columns:")
print(df.columns.tolist())

Columns:
['connection_id', 'source', 'destination', 'mode', 'distance_km', 'duration_min', 'cost_inr', 'co2_kg']


In [6]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
connection_id    0
source           0
destination      0
mode             0
distance_km      0
duration_min     0
cost_inr         0
co2_kg           0
dtype: int64


In [7]:
print('Unique Connections: ',df['connection_id'].nunique())

Unique Connections:  30


In [8]:
locations=pd.unique(df[['source','destination']].values.ravel())
print('Unique locations: ',len(locations))

print('Locations: ')
print(locations)

Unique locations:  19
Locations: 
['Koti' 'Abids' 'Nampally' 'Charminar' 'Mehdipatnam' 'Ameerpet'
 'Banjara Hills' 'Begumpet' 'Secunderabad' 'Jubilee Hills' 'HITEC City'
 'Gachibowli' 'Miyapur' 'Kukatpally' 'Paradise' 'Dilsukhnagar' 'LB Nagar'
 'Uppal' 'Tolichowki']


In [9]:
print('Transportation Modes: ')
print(df['mode'].value_counts())

Transportation Modes: 
mode
Walking             30
Bicycle             30
Public Transport    30
Car                 30
Name: count, dtype: int64


#### Building the Connected Transportation Network

In [10]:
#unique connections
connections=df[['connection_id','source','destination']].drop_duplicates()
connections

,connection_id,source,destination
0,C001,Koti,Abids
4,C002,Koti,Nampally
8,C003,Koti,Charminar
12,C004,Abids,Nampally
16,C005,Abids,Charminar
20,C006,Nampally,Mehdipatnam
24,C007,Nampally,Ameerpet
28,C008,Mehdipatnam,Charminar
32,C009,Mehdipatnam,Banjara Hills
36,C010,Ameerpet,Begumpet


In [11]:
print('Total Connections: ',len(connections))

print('Total Locations: ',len(locations))

Total Connections:  30
Total Locations:  19


In [12]:
### creating network structure

network={}

for index,row in connections.iterrows():
    source=row['source']
    destination=row['destination']
    
    if source not in network:
        network[source]=set()
        
    if destination not in network:
        network[destination]=set()
        
    network[source].add(destination)
    network[destination].add(source)

for location in network:
    network[location]=list(network[location])

In [13]:
### view the network

for location,connected_locations in network.items():
    print(location,'->',connected_locations)

Koti -> ['Charminar', 'Nampally', 'Dilsukhnagar', 'Abids', 'Uppal']
Abids -> ['Charminar', 'Koti', 'Nampally']
Nampally -> ['Abids', 'Ameerpet', 'Mehdipatnam', 'Koti']
Charminar -> ['Abids', 'Mehdipatnam', 'Koti']
Mehdipatnam -> ['Charminar', 'Banjara Hills', 'Nampally']
Ameerpet -> ['Begumpet', 'Banjara Hills', 'Kukatpally', 'Nampally']
Banjara Hills -> ['Jubilee Hills', 'Ameerpet', 'Mehdipatnam']
Begumpet -> ['Paradise', 'Ameerpet', 'Jubilee Hills', 'Secunderabad']
Secunderabad -> ['Paradise', 'Begumpet']
Jubilee Hills -> ['Banjara Hills', 'HITEC City', 'Begumpet', 'Gachibowli']
HITEC City -> ['Kukatpally', 'Jubilee Hills', 'Miyapur', 'Gachibowli']
Gachibowli -> ['Tolichowki', 'Jubilee Hills', 'HITEC City']
Miyapur -> ['Kukatpally', 'HITEC City']
Kukatpally -> ['Ameerpet', 'Miyapur', 'HITEC City']
Paradise -> ['Secunderabad', 'Begumpet']
Dilsukhnagar -> ['LB Nagar', 'Koti']
LB Nagar -> ['Dilsukhnagar', 'Uppal']
Uppal -> ['LB Nagar', 'Koti']
Tolichowki -> ['Gachibowli']


In [14]:
### verify all locations are connected

print('Locations in Network: ',len(network))
print('Locations in Dataset: ',len(locations))

Locations in Network:  19
Locations in Dataset:  19


In [15]:
missing_locations=set(locations)-set(network.keys())

print('Missing Locations: ',missing_locations)

Missing Locations:  set()


In [16]:
### checking direct connections

for location in network:
    print(f'{location}: {len(network[location])} direct connections')

Koti: 5 direct connections
Abids: 3 direct connections
Nampally: 4 direct connections
Charminar: 3 direct connections
Mehdipatnam: 3 direct connections
Ameerpet: 4 direct connections
Banjara Hills: 3 direct connections
Begumpet: 4 direct connections
Secunderabad: 2 direct connections
Jubilee Hills: 4 direct connections
HITEC City: 4 direct connections
Gachibowli: 3 direct connections
Miyapur: 2 direct connections
Kukatpally: 3 direct connections
Paradise: 2 direct connections
Dilsukhnagar: 2 direct connections
LB Nagar: 2 direct connections
Uppal: 2 direct connections
Tolichowki: 1 direct connections


In [17]:
###check for isolated locations

isolated_locations=[location for location in network if len(network[location])==0]

print('Isolated locations: ',isolated_locations)

Isolated locations:  []


In [18]:
print('Total Connections: ',len(connections))
print('Total Locations: ',len(network))
print('Isolated Locations: ',isolated_locations)

Total Connections:  30
Total Locations:  19
Isolated Locations:  []


####  Finding Possible Paths


In [19]:
### creating oath-finding function

def find_paths(network,source,destination):
    paths=[]
    
    def search(current,path):
        if current==destination:
            paths.append(path.copy())
            return 
        for next_location in network[current]:
            if  next_location not in path:
                path.append(next_location)
                search(next_location,path)
                path.pop()
    search(source,[source])
    
    return paths
        

### Test Basic path finding

In [20]:
paths=find_paths(network,'Koti','HITEC City')
print('Number of paths found: ',len(paths))

Number of paths found:  78


In [21]:
for i,path in enumerate(paths[:10],start=1):
    print(f'Path {i}:',"->".join(path))

Path 1: Koti->Charminar->Abids->Nampally->Ameerpet->Begumpet->Jubilee Hills->HITEC City
Path 2: Koti->Charminar->Abids->Nampally->Ameerpet->Begumpet->Jubilee Hills->Gachibowli->HITEC City
Path 3: Koti->Charminar->Abids->Nampally->Ameerpet->Banjara Hills->Jubilee Hills->HITEC City
Path 4: Koti->Charminar->Abids->Nampally->Ameerpet->Banjara Hills->Jubilee Hills->Gachibowli->HITEC City
Path 5: Koti->Charminar->Abids->Nampally->Ameerpet->Kukatpally->Miyapur->HITEC City
Path 6: Koti->Charminar->Abids->Nampally->Ameerpet->Kukatpally->HITEC City
Path 7: Koti->Charminar->Abids->Nampally->Mehdipatnam->Banjara Hills->Jubilee Hills->HITEC City
Path 8: Koti->Charminar->Abids->Nampally->Mehdipatnam->Banjara Hills->Jubilee Hills->Begumpet->Ameerpet->Kukatpally->Miyapur->HITEC City
Path 9: Koti->Charminar->Abids->Nampally->Mehdipatnam->Banjara Hills->Jubilee Hills->Begumpet->Ameerpet->Kukatpally->HITEC City
Path 10: Koti->Charminar->Abids->Nampally->Mehdipatnam->Banjara Hills->Jubilee Hills->Gachibow

In [22]:
test_pairs=[('Koti','HITEC City'),('Koti','Charminar'),('Dilsukhnagar','Gachibowli')]

for source,destination in test_pairs:
    test_paths=find_paths(network,source,destination)
    
    print(f'{source}->{destination}')
    print('Number of paths found: ',len(test_paths))
    
    for i,path in enumerate(test_paths[:3],start=1):
        print(f' Path {i}:',"->".join(path))
        
    print()

Koti->HITEC City
Number of paths found:  78
 Path 1: Koti->Charminar->Abids->Nampally->Ameerpet->Begumpet->Jubilee Hills->HITEC City
 Path 2: Koti->Charminar->Abids->Nampally->Ameerpet->Begumpet->Jubilee Hills->Gachibowli->HITEC City
 Path 3: Koti->Charminar->Abids->Nampally->Ameerpet->Banjara Hills->Jubilee Hills->HITEC City

Koti->Charminar
Number of paths found:  17
 Path 1: Koti->Charminar
 Path 2: Koti->Nampally->Abids->Charminar
 Path 3: Koti->Nampally->Ameerpet->Begumpet->Jubilee Hills->Banjara Hills->Mehdipatnam->Charminar

Dilsukhnagar->Gachibowli
Number of paths found:  200
 Path 1: Dilsukhnagar->LB Nagar->Uppal->Koti->Charminar->Abids->Nampally->Ameerpet->Begumpet->Jubilee Hills->HITEC City->Gachibowli
 Path 2: Dilsukhnagar->LB Nagar->Uppal->Koti->Charminar->Abids->Nampally->Ameerpet->Begumpet->Jubilee Hills->Gachibowli
 Path 3: Dilsukhnagar->LB Nagar->Uppal->Koti->Charminar->Abids->Nampally->Ameerpet->Banjara Hills->Jubilee Hills->HITEC City->Gachibowli



In [23]:
### verifying direcct path

direct_path=['Koti','Charminar']
print('Direct path exists: ',direct_path in find_paths(network,'Koti','Charminar'))

Direct path exists:  True


In [24]:
### Verifying Intermediate Path

paths_koti_hitech=find_paths(network,'Koti','HITEC City')

intermediate_paths=[
    path for path in paths_koti_hitech
    if len(path)>5
]

print('Intermediate paths found: ',len(intermediate_paths))

Intermediate paths found:  77


###  Generating Multimodal Journeys

In [39]:
### Define Available Transportation Modes
modes = [
    "Walking",
    "Bicycle",
    "Public Transport",
    "Car"
]

print("Available Modes:", modes)

Available Modes: ['Walking', 'Bicycle', 'Public Transport', 'Car']


In [41]:
###  Identifying Path Segments
def get_segments(path):
    segments = []

    for i in range(len(path) - 1):
        segments.append((path[i], path[i + 1]))

    return segments


In [42]:
### Finding Available Modes for Each Segment

def get_segment_options(path, df):
    segment_options = []

    segments = get_segments(path)

    for source, destination in segments:
        options = df[
            (
                (df["source"] == source) &
                (df["destination"] == destination)
            )
            |
            (
                (df["source"] == destination) &
                (df["destination"] == source)
            )
        ].copy()

        segment_options.append(options)

    return segment_options

In [43]:
### Generating Transportation Options for Each Segment

def get_available_modes(segment_options, available_modes):
    mode_options = []

    for options in segment_options:
        modes_for_segment = options[
            options["mode"].isin(available_modes)
        ].copy()

        mode_options.append(modes_for_segment)

    return mode_options

In [78]:
# ###Generate Multimodal Journeys

# def generate_multimodal_journeys(path, mode_options):
#     journeys = [
#         {
#             "modes": [],
#             "total_time": 0,
#             "total_cost": 0,
#             "total_co2": 0
#         }
#     ]

#     for options in mode_options:
#         new_journeys = []

#         for journey in journeys:
#             for _, row in options.iterrows():

#                 new_journey = {
#                     "modes": journey["modes"] + [row["mode"]],
#                     "total_time": journey["total_time"] + row["duration_min"],
#                     "total_cost": journey["total_cost"] + row["cost_inr"],
#                     "total_co2": journey["total_co2"] + row["co2_kg"]
#                 }

#                 new_journeys.append(new_journey)

#         journeys = new_journeys

#     results = []

#     for journey in journeys:
#         results.append({
#             "path": path,
#             "modes": journey["modes"],
#             "total_time": journey["total_time"],
#             "total_cost": journey["total_cost"],
#             "total_co2": journey["total_co2"]
#         })

#     return pd.DataFrame(results)

In [77]:
# def generate_multimodal_journeys(
#     path,
#     mode_options,
#     max_time,
#     max_budget
# ):
#     journeys = [
#         {
#             "modes": [],
#             "total_time": 0,
#             "total_cost": 0,
#             "total_co2": 0
#         }
#     ]

#     for options in mode_options:
#         new_journeys = []

#         for journey in journeys:
#             for _, row in options.iterrows():

#                 new_time = (
#                     journey["total_time"]
#                     + row["duration_min"]
#                 )

#                 new_cost = (
#                     journey["total_cost"]
#                     + row["cost_inr"]
#                 )

#                 if exceeds_time_limit(new_time, max_time):
#                     continue

#                 if exceeds_budget_limit(new_cost, max_budget):
#                     continue

#                 new_journey = {
#                     "modes": journey["modes"] + [row["mode"]],
#                     "total_time": new_time,
#                     "total_cost": new_cost,
#                     "total_co2": (
#                         journey["total_co2"]
#                         + row["co2_kg"]
#                     )
#                 }

#                 new_journeys.append(new_journey)

#         journeys = new_journeys

#         if not journeys:
#             break

#     results = []

#     for journey in journeys:
#         results.append({
#             "path": path,
#             "modes": journey["modes"],
#             "total_time": journey["total_time"],
#             "total_cost": journey["total_cost"],
#             "total_co2": journey["total_co2"]
#         })

#     return pd.DataFrame(results)

In [102]:
def generate_multimodal_journeys(
    path,
    mode_options,
    max_time,
    max_budget
):
    journeys = [
        {
            "modes": [],
            "total_time": 0,
            "total_cost": 0,
            "total_co2": 0
        }
    ]

    for options in mode_options:
        new_journeys = []

        for journey in journeys:
            for _, row in options.iterrows():

                new_time = (
                    journey["total_time"]
                    + row["duration_min"]
                )

                new_cost = (
                    journey["total_cost"]
                    + row["cost_inr"]
                )

                if exceeds_time_limit(new_time, max_time):
                    continue

                if exceeds_budget_limit(new_cost, max_budget):
                    continue

                new_journey = {
                    "modes": journey["modes"] + [row["mode"]],
                    "total_time": new_time,
                    "total_cost": new_cost,
                    "total_co2": (
                        journey["total_co2"]
                        + row["co2_kg"]
                    )
                }

                new_journeys.append(new_journey)

        journeys = prune_dominated_journeys(new_journeys)

        if not journeys:
            break

    results = []

    for journey in journeys:
        results.append({
            "path": path,
            "modes": journey["modes"],
            "total_time": journey["total_time"],
            "total_cost": journey["total_cost"],
            "total_co2": journey["total_co2"]
        })

    return pd.DataFrame(results)

In [76]:
### Testing Multimodal Journey Generation

journeys = generate_multimodal_journeys(
    paths[0],
    get_available_modes(
        get_segment_options(paths[0], df),
        modes
    )
)

print("Number of journeys:", len(journeys))
journeys.head()

Number of journeys: 16384


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",341,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",290,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",307,25,0.6
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",284,90,1.2
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",298,0,0.0


In [74]:
### Validating Generated Journey Metrics

print("Time range:")
print(journeys["total_time"].min(),"to",journeys["total_time"].max(),"minutes")

print()
print("Cost range:")
print(journeys["total_cost"].min(),"to",journeys["total_cost"].max(),"INR")

print()
print("CO₂ range:")
print(journeys["total_co2"].min(),"to",journeys["total_co2"].max(),"kg")

print()
print("Missing Values:")
print(journeys[["total_time", "total_cost", "total_co2"]].isnull().sum())

Time range:
93 to 341 minutes

Cost range:
0 to 415 INR

CO₂ range:
0.0 to 5.6000000000000005 kg

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


In [58]:
### Confirming Generated Journey Structure

print("Journey columns:")
print(journeys.columns.tolist())

print("Number of journeys:", len(journeys))

journeys.head()

Journey columns:
['path', 'modes', 'total_time', 'total_cost', 'total_co2']
Number of journeys: 16384


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",341,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",290,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",307,25,0.6
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",284,90,1.2
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Walking, Walking, Walking, Walking, Walking, ...",298,0,0.0


### Efficient Journey Generation

In [86]:
###  Filtering Transportation Modes

def filter_by_modes(df, available_modes):
    filtered_df = df[
        df["mode"].isin(available_modes)
    ].copy()

    return filtered_df

In [87]:
###Early Time Pruning

def exceeds_time_limit(current_time, max_time):
    return current_time > max_time

In [88]:
### Early Budget Pruning.

def exceeds_budget_limit(current_cost, max_budget):
    return current_cost > max_budget

In [107]:
### Dominance Pruning

def is_dominated(journey, other_journey):
    return (
        other_journey["total_time"] <= journey["total_time"]
        and
        other_journey["total_cost"] <= journey["total_cost"]
        and
        other_journey["total_co2"] <= journey["total_co2"]
        and
        (
            other_journey["total_time"] < journey["total_time"]
            or
            other_journey["total_cost"] < journey["total_cost"]
            or
            other_journey["total_co2"] < journey["total_co2"]
        )
    )

In [90]:
### Path Feasibility Check

def is_path_feasible(path, df, available_modes, max_time, max_budget):
    total_min_time = 0
    total_min_cost = 0

    for i in range(len(path) - 1):
        source = path[i]
        destination = path[i + 1]

        options = df[
            (
                (df["source"] == source) &
                (df["destination"] == destination)
            )
            |
            (
                (df["source"] == destination) &
                (df["destination"] == source)
            )
        ]

        options = options[
            options["mode"].isin(available_modes)
        ]

        if options.empty:
            return False

        total_min_time += options["duration_min"].min()
        total_min_cost += options["cost_inr"].min()

    return (
        total_min_time <= max_time
        and
        total_min_cost <= max_budget
    )

In [91]:
### Process Feasible Paths

def process_feasible_paths(
    paths,
    df,
    available_modes,
    max_time,
    max_budget
):
    feasible_paths = []

    for path in paths:
        if is_path_feasible(
            path,
            df,
            available_modes,
            max_time,
            max_budget
        ):
            feasible_paths.append(path)

    return feasible_paths

In [92]:
###  Generating Journeys for Feasible Paths

def generate_feasible_journeys(
    feasible_paths,
    df,
    available_modes
):
    all_journeys = []

    filtered_df = filter_by_modes(
        df,
        available_modes
    )

    for path in feasible_paths:
        segment_options = get_segment_options(
            path,
            filtered_df
        )

        mode_options = get_available_modes(
            segment_options,
            available_modes
        )

        path_journeys = generate_multimodal_journeys(
            path,
            mode_options
        )

        all_journeys.append(path_journeys)

    if not all_journeys:
        return pd.DataFrame(
            columns=[
                "path",
                "modes",
                "total_time",
                "total_cost",
                "total_co2"
            ]
        )

    return pd.concat(
        all_journeys,
        ignore_index=True
    )

In [93]:
###Update the Feasible-Journey Generator

def generate_feasible_journeys(
    feasible_paths,
    df,
    available_modes,
    max_time,
    max_budget
):
    all_journeys = []

    filtered_df = filter_by_modes(
        df,
        available_modes
    )

    for path in feasible_paths:
        segment_options = get_segment_options(
            path,
            filtered_df
        )

        mode_options = get_available_modes(
            segment_options,
            available_modes
        )

        path_journeys = generate_multimodal_journeys(
            path,
            mode_options,
            max_time,
            max_budget
        )

        if not path_journeys.empty:
            all_journeys.append(path_journeys)

    if not all_journeys:
        return pd.DataFrame(
            columns=[
                "path",
                "modes",
                "total_time",
                "total_cost",
                "total_co2"
            ]
        )

    return pd.concat(
        all_journeys,
        ignore_index=True
    )

In [94]:
### Testing the Optimized Journey Generation

test_available_modes = [
    "Walking",
    "Bicycle",
    "Public Transport"
]

test_max_time = 120
test_max_budget = 300

filtered_test_df = filter_by_modes(
    df,
    test_available_modes
)

print("Filtered Dataset Rows:", len(filtered_test_df))
print("Available Modes:", test_available_modes)
print("Maximum Time:", test_max_time, "minutes")
print("Maximum Budget:", test_max_budget, "INR")


Filtered Dataset Rows: 90
Available Modes: ['Walking', 'Bicycle', 'Public Transport']
Maximum Time: 120 minutes
Maximum Budget: 300 INR


In [97]:
#### Creating Feasible Paths

feasible_paths = process_feasible_paths(
    paths,
    df,
    test_available_modes,
    test_max_time,
    test_max_budget
)

print("Total Paths:", len(paths))
print("Feasible Paths:", len(feasible_paths))

Total Paths: 78
Feasible Paths: 26


In [99]:
#### Generate feasible journeys  
feasible_journeys = generate_feasible_journeys(
    feasible_paths,
    df,
    test_available_modes,
    test_max_time,
    test_max_budget
)

print("Number of Feasible Journeys:", len(feasible_journeys))

feasible_journeys.head()

Number of Feasible Journeys: 467


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Public Transport, Bicycle, ...",120,15,0.2
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",118,15,0.3
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Public Transport, ...",120,20,0.4


In [100]:
### Apply Dominance Pruning.

def prune_dominated_journeys(journeys):
    non_dominated = []

    for journey in journeys:
        dominated = False

        for other_journey in journeys:
            if journey is other_journey:
                continue

            if is_dominated(journey, other_journey):
                dominated = True
                break

        if not dominated:
            non_dominated.append(journey)

    return non_dominated

In [103]:
### Test the Optimized Journey Generation

test_path = feasible_paths[0]

segment_options = get_segment_options(
    test_path,
    filter_by_modes(df, test_available_modes)
)

mode_options = get_available_modes(
    segment_options,
    test_available_modes
)

test_journeys = generate_multimodal_journeys(
    test_path,
    mode_options,
    test_max_time,
    test_max_budget
)

print("Test Path:", " → ".join(test_path))
print("Feasible Journeys:", len(test_journeys))

test_journeys.head()

Test Path: Koti → Charminar → Abids → Nampally → Ameerpet → Begumpet → Jubilee Hills → HITEC City
Feasible Journeys: 1


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0


In [104]:
### Validating Optimized Journeys

print("Maximum Generated Time:",
      test_journeys["total_time"].max())

print("Maximum Generated Cost:",
      test_journeys["total_cost"].max())

print("\nTime Constraint Satisfied:",
      (test_journeys["total_time"] <= test_max_time).all())

print("Budget Constraint Satisfied:",
      (test_journeys["total_cost"] <= test_max_budget).all())

Maximum Generated Time: 115
Maximum Generated Cost: 0

Time Constraint Satisfied: True
Budget Constraint Satisfied: True


In [105]:
### Process All Feasible Paths

feasible_journeys = generate_feasible_journeys(
    feasible_paths,
    df,
    test_available_modes,
    test_max_time,
    test_max_budget
)

print("Total Feasible Paths:", len(feasible_paths))
print("Total Feasible Journeys:", len(feasible_journeys))

feasible_journeys.head()

Total Feasible Paths: 26
Total Feasible Journeys: 36


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",120,0,0.0
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, K...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
4,"[Koti, Charminar, Abids, Nampally, Mehdipatnam...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",114,0,0.0


In [106]:
### Validate the Full Optimized Results

print("Total Feasible Journeys:", len(feasible_journeys))

print("\nColumns:")
print(feasible_journeys.columns.tolist())

print("\nMissing Values:")
print(
    feasible_journeys[
        ["total_time", "total_cost", "total_co2"]
    ].isnull().sum()
)

print("\nTime Constraint Satisfied:",
      (feasible_journeys["total_time"] <= test_max_time).all())

print("Budget Constraint Satisfied:",
      (feasible_journeys["total_cost"] <= test_max_budget).all())

Total Feasible Journeys: 36

Columns:
['path', 'modes', 'total_time', 'total_cost', 'total_co2']

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64

Time Constraint Satisfied: True
Budget Constraint Satisfied: True


### Route Evaluation

In [109]:
## Evaluating Journey Metrics

print("Journey Evaluation Summary")

print("\nTravel Time:")
print(
    feasible_journeys["total_time"].min(),
    "to",
    feasible_journeys["total_time"].max(),
    "minutes"
)

print("\nTravel Cost:")
print(
    feasible_journeys["total_cost"].min(),
    "to",
    feasible_journeys["total_cost"].max(),
    "INR"
)

print("\nCO₂ Emissions:")
print(
    feasible_journeys["total_co2"].min(),
    "to",
    feasible_journeys["total_co2"].max(),
    "kg"
)

Journey Evaluation Summary

Travel Time:
85 to 120 minutes

Travel Cost:
0 to 0 INR

CO₂ Emissions:
0.0 to 0.0 kg


In [110]:
### Validate Journey Values

print("Missing Values:")
print(
    feasible_journeys[
        ["total_time", "total_cost", "total_co2"]
    ].isnull().sum()
)

print("\nNumeric Data Types:")
print(
    feasible_journeys[
        ["total_time", "total_cost", "total_co2"]
    ].dtypes
)

print("\nNegative Values:")
print(
    (feasible_journeys[
        ["total_time", "total_cost", "total_co2"]
    ] < 0).sum()
)

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64

Numeric Data Types:
total_time      int64
total_cost      int64
total_co2     float64
dtype: object

Negative Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


In [111]:
## Preparing Evaluated Journeys

evaluated_journeys = feasible_journeys[
    [
        "path",
        "modes",
        "total_time",
        "total_cost",
        "total_co2"
    ]
].copy()

print("Evaluated Journeys:", len(evaluated_journeys))

evaluated_journeys.head()

Evaluated Journeys: 36


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",120,0,0.0
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, K...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
4,"[Koti, Charminar, Abids, Nampally, Mehdipatnam...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",114,0,0.0


### User Constraints & Route Planning

In [112]:
## Create the Route Planning Function

def plan_journeys(
    source,
    destination,
    max_budget,
    max_time,
    available_modes
):
    paths = find_paths(
        network,
        source,
        destination
    )

    feasible_paths = process_feasible_paths(
        paths,
        df,
        available_modes,
        max_time,
        max_budget
    )

    journeys = generate_feasible_journeys(
        feasible_paths,
        df,
        available_modes,
        max_time,
        max_budget
    )

    return journeys

In [113]:
## Testing the Route Planner

test_planned_journeys = plan_journeys(
    source=feasible_paths[0][0],
    destination=feasible_paths[0][-1],
    max_budget=test_max_budget,
    max_time=test_max_time,
    available_modes=test_available_modes
)

print("Number of Planned Journeys:", len(test_planned_journeys))

test_planned_journeys.head()

Number of Planned Journeys: 36


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",120,0,0.0
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, K...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
4,"[Koti, Charminar, Abids, Nampally, Mehdipatnam...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",114,0,0.0


In [114]:
## Validate the Route Planner

print("Maximum Generated Time:",
      test_planned_journeys["total_time"].max(),
      "minutes")

print("Maximum Generated Cost:",
      test_planned_journeys["total_cost"].max(),
      "INR")

print("\nTime Constraint Satisfied:",
      (test_planned_journeys["total_time"] <= test_max_time).all())

print("Budget Constraint Satisfied:",
      (test_planned_journeys["total_cost"] <= test_max_budget).all())

Maximum Generated Time: 120 minutes
Maximum Generated Cost: 0 INR

Time Constraint Satisfied: True
Budget Constraint Satisfied: True


In [115]:
### Test Different User Inputs

test_planned_journeys_2 = plan_journeys(
    source=feasible_paths[0][0],
    destination=feasible_paths[0][-1],
    max_budget=500,
    max_time=150,
    available_modes=[
        "Walking",
        "Public Transport",
        "Car"
    ]
)

print("Number of Planned Journeys:",
      len(test_planned_journeys_2))

test_planned_journeys_2.head()

Number of Planned Journeys: 1567


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Public Transport, Public Transport, Walking, ...",146,270,3.9
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Public Transport, Public Transport, Walking, ...",144,270,4.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Public Transport, Public Transport, Walking, ...",148,260,3.8
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Public Transport, Public Transport, Walking, ...",131,310,4.3
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Public Transport, Public Transport, Public Tr...",139,285,4.1


In [117]:
### Confirm User Input Parameters

print("Route Planner Inputs:")

print("Source:", feasible_paths[0][0])
print("Destination:", feasible_paths[0][-1])
print("Maximum Budget:", 500, "INR")
print("Maximum Travel Time:", 150, "minutes")
print(
    "Available Modes:",
    ["Walking", "Public Transport", "Car"]
)

Route Planner Inputs:
Source: Koti
Destination: HITEC City
Maximum Budget: 500 INR
Maximum Travel Time: 150 minutes
Available Modes: ['Walking', 'Public Transport', 'Car']


### Dominated Route Removal

In [118]:
### Preparing Routes for Dominance Checking

def remove_dominated_routes(journeys):
    non_dominated = []

    for _, journey in journeys.iterrows():
        dominated = False

        for _, other_journey in journeys.iterrows():

            if is_dominated(
                journey,
                other_journey
            ):
                dominated = True
                break

        if not dominated:
            non_dominated.append(journey)

    if not non_dominated:
        return pd.DataFrame(
            columns=[
                "path",
                "modes",
                "total_time",
                "total_cost",
                "total_co2"
            ]
        )

    return pd.DataFrame(non_dominated).reset_index(drop=True)

In [119]:
### Apply Dominance Rule

non_dominated_journeys = remove_dominated_routes(
    test_planned_journeys_2
)

print("Original Journeys:",
      len(test_planned_journeys_2))

print("Non-Dominated Journeys:",
      len(non_dominated_journeys))

non_dominated_journeys.head()

Original Journeys: 1567
Non-Dominated Journeys: 25


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Public Transport, Public Tr...",132,130,2.3
1,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Public Transport, Public Tr...",109,195,2.9
2,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Public Transport, Publ...",128,135,2.4
3,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Public Transport, Publ...",105,200,3.0
4,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Car, Public Transport,...",90,245,3.4


In [120]:
## Validating Non-Dominated Routes

for i, journey in non_dominated_journeys.iterrows():

    dominated = False

    for j, other_journey in non_dominated_journeys.iterrows():

        if i == j:
            continue

        if is_dominated(journey, other_journey):
            dominated = True
            break

    if dominated:
        print("Dominated route found.")
        break

else:
    print("All remaining routes are non-dominated.")

All remaining routes are non-dominated.


In [121]:
### Prepare Non-Dominated Routes

dominated_free_journeys = non_dominated_journeys[
    [
        "path",
        "modes",
        "total_time",
        "total_cost",
        "total_co2"
    ]
].copy()

print("Non-Dominated Journeys:",
      len(dominated_free_journeys))

dominated_free_journeys.head()


Non-Dominated Journeys: 25


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Public Transport, Public Tr...",132,130,2.3
1,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Public Transport, Public Tr...",109,195,2.9
2,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Public Transport, Publ...",128,135,2.4
3,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Public Transport, Publ...",105,200,3.0
4,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Car, Public Transport,...",90,245,3.4


In [122]:
### Validating Dominated-Route Removal

print("Final Non-Dominated Journeys:",
      len(dominated_free_journeys))

print("\nColumns:")
print(dominated_free_journeys.columns.tolist())

print("\nMissing Values:")
print(
    dominated_free_journeys[
        ["total_time", "total_cost", "total_co2"]
    ].isnull().sum()
)

Final Non-Dominated Journeys: 25

Columns:
['path', 'modes', 'total_time', 'total_cost', 'total_co2']

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


### Top 3 Route Selection.

In [123]:
### Prepare Candidate Routes

candidate_routes = dominated_free_journeys.copy()

print("Candidate Routes:", len(candidate_routes))

candidate_routes.head()

Candidate Routes: 25


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Public Transport, Public Tr...",132,130,2.3
1,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Public Transport, Public Tr...",109,195,2.9
2,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Public Transport, Publ...",128,135,2.4
3,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Public Transport, Publ...",105,200,3.0
4,"[Koti, Nampally, Ameerpet, Banjara Hills, Jubi...","[Public Transport, Car, Car, Public Transport,...",90,245,3.4


In [124]:
### Selecting Up to 3 Routes

candidate_routes = candidate_routes.copy()

candidate_routes["route_score"] = (
    candidate_routes["total_time"].rank(pct=True)
    + candidate_routes["total_cost"].rank(pct=True)
    + candidate_routes["total_co2"].rank(pct=True)
)

top_routes = (
    candidate_routes
    .sort_values("route_score")
    .head(3)
    .drop(columns="route_score")
    .reset_index(drop=True)
)

print("Selected Routes:", len(top_routes))

top_routes

Selected Routes: 3


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Public Transport, Public Tr...",140,85,2.0
1,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Car, Public Transport, Public Transport, Publ...",129,115,2.4
2,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Car, Public Transport, Publ...",125,125,2.4


In [127]:
###  Validating the Selected Routes

print("Number of Selected Routes:", len(top_routes))

print("\nColumns:")
print(top_routes.columns.tolist())

print("\nMissing Values:")
print(
    top_routes[
        ["total_time", "total_cost", "total_co2"]
    ].isnull().sum()
)

Number of Selected Routes: 3

Columns:
['path', 'modes', 'total_time', 'total_cost', 'total_co2']

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


In [128]:
### Preparing Final Route Options

final_route_options = top_routes[
    [
        "path",
        "modes",
        "total_time",
        "total_cost",
        "total_co2"
    ]
].copy()

print("Final Route Options:", len(final_route_options))

final_route_options

Final Route Options: 3


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Public Transport, Public Tr...",140,85,2.0
1,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Car, Public Transport, Public Transport, Publ...",129,115,2.4
2,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Car, Public Transport, Publ...",125,125,2.4


In [129]:
### Validating Final Route Options

print("Final Route Options:", len(final_route_options))

print("\nMaximum Routes Allowed:",
      len(final_route_options) <= 3)

print("\nRequired Columns Present:",
      set([
          "path",
          "modes",
          "total_time",
          "total_cost",
          "total_co2"
      ]).issubset(final_route_options.columns))

print("\nMissing Values:")
print(
    final_route_options[
        ["total_time", "total_cost", "total_co2"]
    ].isnull().sum()
)

Final Route Options: 3

Maximum Routes Allowed: True

Required Columns Present: True

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


### No-Feasible-Route Handling

In [130]:
def generate_alternative_routes(
    source,
    destination,
    df,
    available_modes
):
    paths = find_paths(
        network,
        source,
        destination
    )

    alternative_journeys = []

    filtered_df = filter_by_modes(
        df,
        available_modes
    )

    for path in paths:

        segment_options = get_segment_options(
            path,
            filtered_df
        )

        mode_options = get_available_modes(
            segment_options,
            available_modes
        )

        if any(options.empty for options in mode_options):
            continue

        journeys = generate_multimodal_journeys(
            path,
            mode_options,
            float("inf"),
            float("inf")
        )

        if not journeys.empty:
            alternative_journeys.append(journeys)

    if not alternative_journeys:
        return pd.DataFrame(
            columns=[
                "path",
                "modes",
                "total_time",
                "total_cost",
                "total_co2"
            ]
        )

    return pd.concat(
        alternative_journeys,
        ignore_index=True
    )

In [131]:
### Test a No-Feasible-Route Scenario

no_feasible_routes = generate_alternative_routes(
    source=feasible_paths[0][0],
    destination=feasible_paths[0][-1],
    df=df,
    available_modes=test_available_modes
)

print("Alternative Journeys:", len(no_feasible_routes))
no_feasible_routes.head()

Alternative Journeys: 126


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",126,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",120,0,0.0
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, K...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",138,0,0.0


In [132]:
### Calculate how far alternatives are from the user's limits

alternative_routes = no_feasible_routes.copy()

alternative_routes["extra_time"] = (
    alternative_routes["total_time"] - test_max_time
).clip(lower=0)

alternative_routes["extra_budget"] = (
    alternative_routes["total_cost"] - test_max_budget
).clip(lower=0)

alternative_routes["feasible"] = (
    (alternative_routes["total_time"] <= test_max_time)
    &
    (alternative_routes["total_cost"] <= test_max_budget)
)

print("Total Alternatives:", len(alternative_routes))

print(
    "Feasible Routes:",
    alternative_routes["feasible"].sum()
)

print(
    "Infeasible Alternatives:",
    (~alternative_routes["feasible"]).sum()
)

alternative_routes.head()

Total Alternatives: 126
Feasible Routes: 36
Infeasible Alternatives: 90


,path,modes,total_time,total_cost,total_co2,extra_time,extra_budget,feasible
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0,0,0,True
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",126,0,0.0,6,0,False
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0,0,0,True
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",120,0,0.0,0,0,True
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, K...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",138,0,0.0,18,0,False


In [133]:
### Select the Closest Infeasible Alternatives

def get_closest_alternatives(
    alternative_routes,
    max_time,
    max_budget,
    number_of_routes=3
):
    alternatives = alternative_routes.copy()

    alternatives = alternatives[
        ~alternatives["feasible"]
    ].copy()

    if alternatives.empty:
        return alternatives

    alternatives["alternative_score"] = (
        alternatives["extra_time"].rank(pct=True)
        + alternatives["extra_budget"].rank(pct=True)
        + alternatives["total_co2"].rank(pct=True)
    )

    closest_alternatives = (
        alternatives
        .sort_values("alternative_score")
        .head(number_of_routes)
        .drop(columns="alternative_score")
        .reset_index(drop=True)
    )

    return closest_alternatives

In [134]:
closest_alternatives = get_closest_alternatives(
    alternative_routes,
    test_max_time,
    test_max_budget
)

print(
    "Closest Infeasible Alternatives:",
    len(closest_alternatives)
)

closest_alternatives

Closest Infeasible Alternatives: 3


,path,modes,total_time,total_cost,total_co2,extra_time,extra_budget,feasible
0,"[Koti, Nampally, Mehdipatnam, Banjara Hills, A...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",124,0,0.0,4,0,False
1,"[Koti, Nampally, Mehdipatnam, Banjara Hills, A...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",124,0,0.0,4,0,False
2,"[Koti, Nampally, Abids, Charminar, Mehdipatnam...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",124,0,0.0,4,0,False


In [135]:
### Prepare the Fallback Display

fallback_routes = closest_alternatives[
    [
        "path",
        "modes",
        "total_time",
        "total_cost",
        "total_co2",
        "extra_time",
        "extra_budget"
    ]
].copy()

print("Fallback Route Options:", len(fallback_routes))
fallback_routes

Fallback Route Options: 3


,path,modes,total_time,total_cost,total_co2,extra_time,extra_budget
0,"[Koti, Nampally, Mehdipatnam, Banjara Hills, A...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",124,0,0.0,4,0
1,"[Koti, Nampally, Mehdipatnam, Banjara Hills, A...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",124,0,0.0,4,0
2,"[Koti, Nampally, Abids, Charminar, Mehdipatnam...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",124,0,0.0,4,0


In [136]:
### validation cell

print("Fallback Route Validation")

print("\nMaximum Alternatives:",
      len(fallback_routes) <= 3)

print("\nRequired Columns Present:",
      set([
          "path",
          "modes",
          "total_time",
          "total_cost",
          "total_co2",
          "extra_time",
          "extra_budget"
      ]).issubset(fallback_routes.columns))

print("\nMissing Values:")
print(
    fallback_routes[
        [
            "total_time",
            "total_cost",
            "total_co2",
            "extra_time",
            "extra_budget"
        ]
    ].isnull().sum()
)

Fallback Route Validation

Maximum Alternatives: True

Required Columns Present: True

Missing Values:
total_time      0
total_cost      0
total_co2       0
extra_time      0
extra_budget    0
dtype: int64


### AI Recommendation

In [137]:
### Create a Recommendation Score

def calculate_recommendation_score(journeys):
    recommendations = journeys.copy()

    recommendations["recommendation_score"] = (
        recommendations["total_time"].rank(pct=True)
        + recommendations["total_cost"].rank(pct=True)
        + recommendations["total_co2"].rank(pct=True)
    )

    return recommendations

In [138]:
### Apply the Score

scored_routes = calculate_recommendation_score(
    final_route_options
)

scored_routes

,path,modes,total_time,total_cost,total_co2,recommendation_score
0,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Public Transport, Public Tr...",140,85,2.0,1.666667
1,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Car, Public Transport, Public Transport, Publ...",129,115,2.4,2.166667
2,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Car, Public Transport, Publ...",125,125,2.4,2.166667


In [139]:
### Select the Recommended Route

recommended_route = (
    scored_routes
    .sort_values("recommendation_score")
    .head(1)
    .reset_index(drop=True)
)

recommended_route

,path,modes,total_time,total_cost,total_co2,recommendation_score
0,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Public Transport, Public Tr...",140,85,2.0,1.666667


In [142]:
### Generate an Explanation

recommended_route = recommended_route.copy()

recommended_route["recommendation_reason"] = (
    "This route was recommended because it provides "
    "a balanced combination of travel time, cost, and CO2 emissions "
    "among the available feasible routes."
)

recommended_route[
    [
        "path",
        "modes",
        "total_time",
        "total_cost",
        "total_co2",
        "recommendation_reason"
    ]
]

,path,modes,total_time,total_cost,total_co2,recommendation_reason
0,"[Koti, Nampally, Ameerpet, Kukatpally, HITEC C...","[Public Transport, Public Transport, Public Tr...",140,85,2.0,This route was recommended because it provides...


## Final Testing

In [144]:
### Test a Complete Feasible Scenario

## Testcase 1

final_test_journeys = plan_journeys(
    source="Koti",
    destination="HITEC City",
    max_budget=300,
    max_time=150,
    available_modes=[
        "Walking",
        "Bicycle",
        "Public Transport"
    ]
)

print("Final Test Journeys:", len(final_test_journeys))
final_test_journeys.head()

Final Test Journeys: 75


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",115,0,0.0
1,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",126,0,0.0
2,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0
3,"[Koti, Charminar, Abids, Nampally, Ameerpet, B...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",120,0,0.0
4,"[Koti, Charminar, Abids, Nampally, Ameerpet, K...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",138,0,0.0


In [148]:
### Test Case 2

second_test_journeys = plan_journeys(
    source="Dilsukhnagar",
    destination="Gachibowli",
    max_budget=200,
    max_time=180,
    available_modes=[
        "Walking",
        "Bicycle",
        "Public Transport"
    ]
)

print("Second Test Journeys:", len(second_test_journeys))
second_test_journeys.head()

Second Test Journeys: 112


,path,modes,total_time,total_cost,total_co2
0,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",178,0,0.0
1,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",172,0,0.0
2,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",177,0,0.0
3,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",177,0,0.0
4,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",179,0,0.0


In [149]:
### validate

print("Time Constraint Satisfied:",
      (second_test_journeys["total_time"] <= 180).all())

print("Budget Constraint Satisfied:",
      (second_test_journeys["total_cost"] <= 200).all())

print("\nMissing Values:")
print(
    second_test_journeys[
        ["total_time", "total_cost", "total_co2"]
    ].isnull().sum()
)

Time Constraint Satisfied: True
Budget Constraint Satisfied: True

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


In [150]:
### Test No-Feasible-Route Scenario

strict_source = "Dilsukhnagar"
strict_destination = "Gachibowli"
strict_budget = 10
strict_time = 30

strict_modes = [
    "Walking",
    "Bicycle",
    "Public Transport"
]

strict_journeys = plan_journeys(
    source=strict_source,
    destination=strict_destination,
    max_budget=strict_budget,
    max_time=strict_time,
    available_modes=strict_modes
)

print("Feasible Journeys:", len(strict_journeys))

Feasible Journeys: 0


In [151]:
### Test the Fallback System

strict_alternatives = generate_alternative_routes(
    source=strict_source,
    destination=strict_destination,
    df=df,
    available_modes=strict_modes
)

print("Alternative Journeys:", len(strict_alternatives))
strict_alternatives.head()

Alternative Journeys: 320


,path,modes,total_time,total_cost,total_co2
0,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",197,0,0.0
1,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",178,0,0.0
2,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",191,0,0.0
3,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",172,0,0.0
4,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",251,0,0.0


In [153]:
### Calculate the Strict-Case Alternatives

strict_alternative_routes = strict_alternatives.copy()

strict_alternative_routes["extra_time"] = (
    strict_alternative_routes["total_time"] - strict_time
).clip(lower=0)

strict_alternative_routes["extra_budget"] = (
    strict_alternative_routes["total_cost"] - strict_budget
).clip(lower=0)

strict_alternative_routes["feasible"] = (
    (strict_alternative_routes["total_time"] <= strict_time)
    &
    (strict_alternative_routes["total_cost"] <= strict_budget)
)

print("Total Alternatives:", len(strict_alternative_routes))
print("Feasible Routes:", strict_alternative_routes["feasible"].sum())
print("Infeasible Alternatives:",
      (~strict_alternative_routes["feasible"]).sum())

Total Alternatives: 320
Feasible Routes: 0
Infeasible Alternatives: 320


In [154]:
### Select the Closest 3 Alternatives

strict_closest_alternatives = get_closest_alternatives(
    strict_alternative_routes,
    strict_time,
    strict_budget
)

print(
    "Closest Infeasible Alternatives:",
    len(strict_closest_alternatives)
)

strict_closest_alternatives

Closest Infeasible Alternatives: 3


,path,modes,total_time,total_cost,total_co2,extra_time,extra_budget,feasible
0,"[Dilsukhnagar, Koti, Nampally, Ameerpet, Banja...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",104,0,0.0,74,0,False
1,"[Dilsukhnagar, Koti, Abids, Nampally, Ameerpet...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",107,0,0.0,77,0,False
2,"[Dilsukhnagar, Koti, Nampally, Mehdipatnam, Ba...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",109,0,0.0,79,0,False


In [155]:
### Final Fallback Validation

print("Final Fallback Validation")

print("\nMaximum Alternatives:",
      len(strict_closest_alternatives) <= 3)

print("\nRequired Columns Present:",
      set([
          "path",
          "modes",
          "total_time",
          "total_cost",
          "total_co2"
      ]).issubset(
          strict_closest_alternatives.columns
      ))

print("\nMissing Values:")
print(
    strict_closest_alternatives[
        [
            "total_time",
            "total_cost",
            "total_co2"
        ]
    ].isnull().sum()
)

Final Fallback Validation

Maximum Alternatives: True

Required Columns Present: True

Missing Values:
total_time    0
total_cost    0
total_co2     0
dtype: int64


In [157]:
### Low Budget

low_budget_test = plan_journeys(
    source="Dilsukhnagar",
    destination="Gachibowli",
    max_budget=50,
    max_time=180,
    available_modes=[
        "Walking",
        "Bicycle",
        "Public Transport"
    ]
)

print("Test 2 — Low Budget")
print("Journeys Found:", len(low_budget_test))

low_budget_test.head()

Test 2 — Low Budget
Journeys Found: 112


,path,modes,total_time,total_cost,total_co2
0,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",178,0,0.0
1,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",172,0,0.0
2,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",177,0,0.0
3,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",177,0,0.0
4,"[Dilsukhnagar, LB Nagar, Uppal, Koti, Charmina...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",179,0,0.0


In [158]:
### Short Time Limit

short_time_test = plan_journeys(
    source="Dilsukhnagar",
    destination="Gachibowli",
    max_budget=200,
    max_time=100,
    available_modes=[
        "Walking",
        "Bicycle",
        "Public Transport"
    ]
)

print("Test 3 — Short Time Limit")
print("Journeys Found:", len(short_time_test))

short_time_test.head()

Test 3 — Short Time Limit
Journeys Found: 0


,path,modes,total_time,total_cost,total_co2


In [159]:
#### Only Walking

only_walking_test = plan_journeys(
    source="Dilsukhnagar",
    destination="Gachibowli",
    max_budget=200,
    max_time=180,
    available_modes=[
        "Walking"
    ]
)

print("Test 4 — Only Walking")
print("Journeys Found:", len(only_walking_test))

only_walking_test.head()

Test 4 — Only Walking
Journeys Found: 0


,path,modes,total_time,total_cost,total_co2


In [160]:
### Only Walking

only_walking_test = plan_journeys(
    source="Koti",
    destination="Abids",
    max_budget=100,
    max_time=120,
    available_modes=[
        "Walking"
    ]
)

print("Test 4 — Only Walking")
print("Journeys Found:", len(only_walking_test))

only_walking_test.head()

Test 4 — Only Walking
Journeys Found: 3


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids]","[Walking, Walking]",88,0,0.0
1,"[Koti, Nampally, Abids]","[Walking, Walking]",57,0,0.0
2,"[Koti, Abids]",[Walking],26,0,0.0


In [161]:
### Walking + Bicycle

walking_bicycle_test = plan_journeys(
    source="Koti",
    destination="Abids",
    max_budget=100,
    max_time=120,
    available_modes=[
        "Walking",
        "Bicycle"
    ]
)

print("Test 5 — Walking + Bicycle")
print("Journeys Found:", len(walking_bicycle_test))

walking_bicycle_test.head()

Test 5 — Walking + Bicycle
Journeys Found: 9


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids]","[Bicycle, Bicycle]",30,0,0.0
1,"[Koti, Charminar, Mehdipatnam, Banjara Hills, ...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",100,0,0.0
2,"[Koti, Charminar, Mehdipatnam, Banjara Hills, ...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",100,0,0.0
3,"[Koti, Charminar, Mehdipatnam, Nampally, Abids]","[Bicycle, Bicycle, Bicycle, Bicycle]",71,0,0.0
4,"[Koti, Nampally, Abids]","[Bicycle, Bicycle]",20,0,0.0


In [162]:
### All Modes

all_modes_test = plan_journeys(
    source="Koti",
    destination="Abids",
    max_budget=100,
    max_time=120,
    available_modes=[
        "Walking",
        "Bicycle",
        "Public Transport",
        "Car"
    ]
)

print("Test 6 — All Modes")
print("Journeys Found:", len(all_modes_test))

all_modes_test.head()

Test 6 — All Modes
Journeys Found: 34


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids]","[Bicycle, Bicycle]",30,0,0.0
1,"[Koti, Charminar, Abids]","[Bicycle, Car]",28,45,0.7
2,"[Koti, Charminar, Abids]","[Car, Bicycle]",27,65,0.9
3,"[Koti, Charminar, Mehdipatnam, Banjara Hills, ...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",100,0,0.0
4,"[Koti, Charminar, Mehdipatnam, Banjara Hills, ...","[Bicycle, Bicycle, Bicycle, Bicycle, Bicycle, ...",99,30,0.4


In [164]:
### Direct Route Available

direct_available = direct_route_test[
    direct_route_test["path"].apply(
        lambda path: len(path) == 2
        and path[0] == "Koti"
        and path[1] == "Abids"
    )
]

print("Test 8 — Direct Route Available")
print("Direct Koti → Abids Journeys:", len(direct_available))

direct_available

Test 8 — Direct Route Available
Direct Koti → Abids Journeys: 2


,path,modes,total_time,total_cost,total_co2
32,"[Koti, Abids]",[Bicycle],9,0,0.0
33,"[Koti, Abids]",[Car],8,35,0.5


In [165]:
### Direct vs Indirect Route

direct_routes = direct_route_test[
    direct_route_test["path"].apply(
        lambda path: len(path) == 2
    )
]

indirect_routes = direct_route_test[
    direct_route_test["path"].apply(
        lambda path: len(path) > 2
    )
]

print("Test 9 — Direct vs Indirect Route")
print("Direct Routes:", len(direct_routes))
print("Indirect Routes:", len(indirect_routes))

print("\nDirect Route Example:")
display(direct_routes.head(1))

print("\nIndirect Route Example:")
display(indirect_routes.head(1))

Test 9 — Direct vs Indirect Route
Direct Routes: 2
Indirect Routes: 32

Direct Route Example:


,path,modes,total_time,total_cost,total_co2
32,"[Koti, Abids]",[Bicycle],9,0,0.0



Indirect Route Example:


,path,modes,total_time,total_cost,total_co2
0,"[Koti, Charminar, Abids]","[Bicycle, Bicycle]",30,0,0.0
